# V-ChartText – Chạy thử baseline Mức 1 trên Colab

Trước khi chạy:
1. **Runtime → Change runtime type → T4 GPU.**
2. Trên Drive, mở thư mục được chia sẻ **Ảnh biểu đồ** → chuột phải → *Sắp xếp → Thêm lối tắt* → chọn *Drive của tôi*. Nhờ vậy Colab thấy được ảnh tại `/content/drive/MyDrive/Ảnh biểu đồ`.
3. Tạo GitHub token (Settings → Developer settings → Fine-grained token, quyền *Contents: Read* cho repo) và lưu vào **Secrets** của Colab (biểu tượng chìa khoá bên trái) với tên `GH_TOKEN`, bật *Notebook access*.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

GH_USER = 'rangha26'          # repo thử nghiệm cá nhân; đổi sang 'dangthihoangnganit6' khi chạy repo nhóm
BRANCH  = 'dev'
DRIVE_IMAGES = '/content/drive/MyDrive/Ảnh biểu đồ'
DRIVE_OUT    = '/content/drive/MyDrive/V-ChartText_outputs'   # nơi lưu kết quả để không mất khi Colab ngắt

In [ ]:
import os
token = userdata.get('GH_TOKEN')
if not os.path.exists('/content/V-ChartText'):
    !git clone -q -b {BRANCH} https://{token}@github.com/{GH_USER}/V-ChartText.git /content/V-ChartText
%cd /content/V-ChartText
!git pull -q origin {BRANCH}   # lấy bản prompts.py mới nhất khi chạy lại
!git log --oneline -1

In [ ]:
!pip install -q "transformers>=4.57.0" accelerate timm einops sentencepiece scipy

## 1. Nhập ảnh từ Drive và kiểm tra nhãn
Dùng lại id của các file nhãn đã commit (khớp theo tên file gốc), nên id trên Colab giống id trên máy bạn.

In [ ]:
!python src/prepare_dataset.py --raw "{DRIVE_IMAGES}"
!python eval/validate_labels.py | tail -8

## 2. Qwen3-VL-2B: chạy thử 5 ảnh, mỗi loại biểu đồ 1 ảnh

In [ ]:
# Chạy lại sau khi sửa prompt: chuyển kết quả cũ sang outputs_v0/ (script bỏ qua id đã có trong file output)
!mkdir -p outputs_v0 && mv outputs/*.jsonl outputs_v0/ 2>/dev/null; ls outputs_v0
!mkdir -p "{DRIVE_OUT}/outputs_v0" && cp outputs_v0/*.jsonl "{DRIVE_OUT}/outputs_v0/" 2>/dev/null; true

In [ ]:
IDS = 'bar_001,line_001,pie_001,combo_001,area_001'
MODEL = 'Qwen/Qwen3-VL-2B-Instruct'
!python src/infer_qwen.py --model {MODEL} --task extract  --ids {IDS}
!python src/infer_qwen.py --model {MODEL} --task describe --ids {IDS}

In [ ]:
import json, glob
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('Qwen/Qwen3-VL-2B-Instruct')

def parse_json(s):
    s = s.strip()
    if s.startswith('```'):
        s = s.split('\n', 1)[1].rsplit('```', 1)[0]
    return json.loads(s)

# In đầy đủ output (không cắt), kèm số token và kiểm tra JSON.
# Số token chính xác với Qwen; với Vintern chỉ là ước lượng.
for f in sorted(glob.glob('outputs/*.jsonl')):
    print('=' * 20, f)
    for line in open(f, encoding='utf-8'):
        r = json.loads(line); out = r['output']
        n = r.get('n_new_tokens') or len(tok(out)['input_ids'])
        info = f"{n} token" + ("  ⚠ CHẠM max_new_tokens" if n >= 760 else "")
        if '__extract__' in f:
            try:
                parse_json(out); info += " | JSON hợp lệ"
            except Exception as e:
                info += f" | JSON LỖI: {e}"
        print(f"--- {r['id']} ({r['latency_s']}s, {info})\n{out}\n")

## 3. Vintern-1B-v3.5
Nếu cell dưới báo lỗi khi nạp mô hình (do mã `trust_remote_code` chưa hợp với transformers mới), chạy cell *hạ phiên bản*, rồi **Runtime → Restart session** và chạy lại từ cell `%cd` ở trên (bỏ qua cell cài đặt).

In [ ]:
!python src/infer_vintern.py --task extract  --ids {IDS}
!python src/infer_vintern.py --task describe --ids {IDS}

In [ ]:
# Chỉ chạy khi cell trên lỗi nạp mô hình
!pip install -q "transformers<4.50"

## 4. Chấm điểm (chỉ tính các ảnh đã gán nhãn xong, `status` khác `todo`)

In [ ]:
for f in sorted(glob.glob('outputs/*__extract__*.jsonl')):
    !python eval/score_extract.py "{f}"
for f in sorted(glob.glob('outputs/*__describe__*.jsonl')):
    !python eval/check_describe.py "{f}"

## 5. Lưu kết quả về Drive

In [ ]:
!mkdir -p "{DRIVE_OUT}"
!cp -r outputs eval/results "{DRIVE_OUT}/" 2>/dev/null; ls -R "{DRIVE_OUT}" | head -30

## 6. Chạy toàn bộ bộ dữ liệu (sau khi đã gán nhãn)
Bỏ `--ids`. Script tự bỏ qua ảnh đã chạy, nên nếu Colab ngắt thì copy lại `outputs/` từ Drive vào rồi chạy tiếp.

In [ ]:
# !cp "{DRIVE_OUT}"/outputs/*.jsonl outputs/   # khôi phục khi chạy tiếp
# !python src/infer_qwen.py --model {MODEL} --task extract
# !python src/infer_qwen.py --model {MODEL} --task describe